In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "<your_catalog_name>"  # e.g. "main"
schema  = "<your_schema_name>"   # e.g. "abac_demo"

print(f"✓ Using: {catalog}.{schema}")


# Tutorial 2: Column Masking with ABAC

This tutorial demonstrates how to use ABAC to mask sensitive columns based on governed tags.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the schema to create policies
- `APPLY TAG` privilege on tables and columns
- `ASSIGN` privilege on the governed tags being used
- Ability to set column masks directly (`ALTER TABLE ... ALTER COLUMN ... SET MASK`) — used for VARIANT and STRUCT examples
- Account group referenced: `abac_tut_admins`

## Setup

> **Note:** This tutorial uses `abac_tutorials` as the catalog name. Replace it with your own catalog throughout if needed. You can create one with `CREATE CATALOG abac_tutorials` (requires metastore admin), or use any existing catalog where you have `CREATE SCHEMA` privileges.

In [ ]:
spark.sql(f"""
-- Grant base access so group members can query the tables.
-- ABAC policies handle the fine-grained row/column filtering on top of these grants.
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


### Create account groups

This tutorial references the following account-level groups: `abac_tut_admins`

Account-level groups (required for Unity Catalog / ABAC) must be created in the **Account Console** (Admin > Groups), not via SQL. `CREATE GROUP` in SQL only creates workspace-local groups which are not compatible with ABAC policies.

> If these groups already exist in your account, skip this step.

### Create governed tags

Before running this tutorial, create the following governed tags in the Catalog Explorer UI:
**Catalog** > **Governed Tags** > **Create governed tag**

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_pii` | `ssn, email, name, phone, address` |

> If you have already created these tags (e.g., from a previous tutorial), you can skip this step.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.employees (
  id INT,
  full_name STRING,
  email STRING,
  ssn STRING,
  phone STRING,
  address STRING,
  salary DOUBLE,
  hire_date DATE,
  metadata VARIANT,
  contact_info STRUCT<primary_email: STRING, secondary_email: STRING, phone: STRING>
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.employees VALUES
  (1, 'Alice Johnson',  'alice@acme.com',   '123-45-6789', '555-0101', '123 Main St, NYC',      95000.00, '2020-03-15', parse_json('{{"department": "Engineering", "level": 5, "start_date": "2020-03-15"}}'), named_struct('primary_email', 'alice@acme.com', 'secondary_email', 'alice.j@gmail.com', 'phone', '555-0101')),
  (2, 'Bob Smith',      'bob@acme.com',     '234-56-7890', '555-0202', '456 Oak Ave, LA',       110000.00, '2019-07-22', parse_json('{{"department": "Sales", "level": 4, "start_date": "2019-07-22"}}'), named_struct('primary_email', 'bob@acme.com', 'secondary_email', 'bob.s@gmail.com', 'phone', '555-0202')),
  (3, 'Carol White',    'carol@acme.com',   '345-67-8901', '555-0303', '789 Pine Rd, Chicago',  125000.00, '2018-01-10', parse_json('{{"department": "Engineering", "level": 6, "start_date": "2018-01-10"}}'), named_struct('primary_email', 'carol@acme.com', 'secondary_email', 'carol.w@gmail.com', 'phone', '555-0303')),
  (4, 'David Lee',      'david@acme.com',   '456-78-9012', '555-0404', '321 Elm St, Houston',   88000.00,  '2021-05-18', parse_json('{{"department": "Marketing", "level": 3, "start_date": "2021-05-18"}}'), named_struct('primary_email', 'david@acme.com', 'secondary_email', 'david.l@gmail.com', 'phone', '555-0404')),
  (5, 'Eva Martinez',   'eva@acme.com',     '567-89-0123', '555-0505', '654 Maple Dr, Phoenix', 102000.00, '2020-11-30', parse_json('{{"department": "HR", "level": 4, "start_date": "2020-11-30"}}'), named_struct('primary_email', 'eva@acme.com', 'secondary_email', 'eva.m@gmail.com', 'phone', '555-0505'))
""")


In [ ]:
spark.sql(f"""
-- Tag PII columns
ALTER TABLE {catalog}.{schema}.employees ALTER COLUMN ssn SET TAGS ('abac_tut_pii' = 'ssn')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employees ALTER COLUMN email SET TAGS ('abac_tut_pii' = 'email')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employees ALTER COLUMN full_name SET TAGS ('abac_tut_pii' = 'name')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employees ALTER COLUMN phone SET TAGS ('abac_tut_pii' = 'phone')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employees ALTER COLUMN address SET TAGS ('abac_tut_pii' = 'address')
""")


## Step 1: Basic String Redaction

The simplest mask: replace the entire value with a fixed string.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.redact(val STRING)
RETURNS STRING
RETURN '***REDACTED***'
""")


In [ ]:
spark.sql(f"""
-- Apply redaction to all PII-tagged columns
CREATE OR REPLACE POLICY redact_pii
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_pii') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- All PII columns are redacted; salary and other non-PII columns are visible
SELECT id, full_name, email, ssn, phone, salary FROM {catalog}.{schema}.employees
""")


## Step 2: Partial Reveal (Last 4 of SSN)

For SSN columns, show only the last 4 digits. This is useful when users need to verify identity without seeing the full value.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_ssn(val STRING)
RETURNS STRING
RETURN CONCAT('***-**-', RIGHT(val, 4))
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE POLICY mask_ssn_policy
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_ssn
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('abac_tut_pii', 'ssn') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- SSN shows last 4 digits; other columns are unmasked (redact_pii was dropped)
SELECT id, full_name, email, ssn, salary FROM {catalog}.{schema}.employees
""")


## Step 3: SHA2 Pseudonymization

SHA2 hashing produces a consistent masked value for each input. This is useful for analytics that need to join or group on masked values without revealing the original data.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.pseudonymize(val STRING)
RETURNS STRING
RETURN SHA2(val, 256)
""")


In [ ]:
spark.sql(f"""
-- Apply pseudonymization to email columns
CREATE OR REPLACE POLICY pseudo_email_policy
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.pseudonymize
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('abac_tut_pii', 'email') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Email is now SHA2-hashed; SSN shows last 4; other columns unmasked
SELECT id, full_name, email, ssn, salary FROM {catalog}.{schema}.employees
""")


## Step 4: VARIANT Masking

VARIANT columns can hold any data type. The `flexible_mask` UDF inspects the runtime type using `schema_of_variant()` and returns an appropriate masked value for each type.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.flexible_mask(data VARIANT)
RETURNS VARIANT
RETURN CASE
  WHEN is_account_group_member('abac_tut_admins') THEN data
  ELSE CASE
    -- WHEN schema_of_variant(data) = 'INT'    THEN 0::VARIANT
    WHEN schema_of_variant(data) = 'BIGINT' THEN -1::VARIANT
    WHEN schema_of_variant(data) = 'DATE'   THEN DATE'1970-01-01'::VARIANT
    WHEN schema_of_variant(data) = 'DOUBLE' THEN 0.00::VARIANT
    WHEN schema_of_variant(data) = 'STRING' THEN '"*******"'::VARIANT
    WHEN schema_of_variant(data) LIKE 'OBJECT%' THEN parse_json('{{"masked": true}}')
    WHEN schema_of_variant(data) LIKE 'ARRAY%' THEN parse_json('[]')
    ELSE NULL::VARIANT
  END
END
""")


In [ ]:
spark.sql(f"""
-- Apply flexible mask to the VARIANT metadata column
-- Using a table-level policy for this specific column
ALTER TABLE {catalog}.{schema}.employees
  ALTER COLUMN metadata SET MASK {catalog}.{schema}.flexible_mask
""")


In [ ]:
spark.sql(f"""
-- The VARIANT column is masked with type-appropriate values
-- Cast to STRING for display since raw VARIANT may not render in results
SELECT id, full_name, CAST(metadata AS STRING) AS metadata FROM {catalog}.{schema}.employees
""")


## Step 5: STRUCT Masking

STRUCT columns contain named fields. You can mask individual fields while leaving others visible by returning a new STRUCT with sensitive fields replaced.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_contact_info(
  val STRUCT<primary_email: STRING, secondary_email: STRING, phone: STRING>
)
RETURNS STRUCT<primary_email: STRING, secondary_email: STRING, phone: STRING>
RETURN CASE
  WHEN is_account_group_member('abac_tut_admins') THEN val
  ELSE named_struct(
    'primary_email', '***REDACTED***',
    'secondary_email', '***REDACTED***',
    'phone', CONCAT('***-', RIGHT(val.phone, 4))
  )
END
""")


In [ ]:
spark.sql(f"""
-- Apply STRUCT mask directly to the column
ALTER TABLE {catalog}.{schema}.employees
  ALTER COLUMN contact_info SET MASK {catalog}.{schema}.mask_contact_info
""")


In [ ]:
spark.sql(f"""
-- Emails are redacted, phone shows last 4 digits
SELECT id, full_name, contact_info FROM {catalog}.{schema}.employees
""")


## Step 6: Combining Multiple Mask Policies

You can have different mask strategies for different PII types. Here we combine the SSN partial-reveal, email pseudonymization, and a generic redaction for name/phone/address columns.

In [ ]:
spark.sql(f"""
-- Redact remaining PII columns (name, phone, address)
CREATE OR REPLACE POLICY redact_other_pii
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS (has_tag_value('abac_tut_pii', 'name')
  OR has_tag_value('abac_tut_pii', 'phone')
  OR has_tag_value('abac_tut_pii', 'address')) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- All mask strategies active: SSN=last4, email=SHA2, name/phone/address=redacted
-- salary, hire_date, id visible (not tagged as PII)
SELECT id, full_name, email, ssn, phone, address, salary FROM {catalog}.{schema}.employees
""")


### Delete governed tags

If you no longer need them, delete `abac_pii` from the Catalog Explorer UI (**Governed Tags** section).

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced (`abac_tut_admins`, `pci_analysts`, `finance_ops`, `abac_tut_admins`, `clinical_staff`, `billing_staff`) must be created in the **Account Console** by an admin.
>
> Governed tags `pci_data` (with values: credit_card, ssn, income, phone, email) and `phi_level` (with values: name, dob, id, contact, diagnosis) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Financial Services — PCI-DSS Compliant Column Masking

Payment Card Industry Data Security Standard (PCI-DSS) mandates strict controls over cardholder data. Credit card numbers, CVVs, and SSNs must be masked for all users except authorized personnel.

**Key PCI-DSS requirements addressed:**
- Requirement 3.3: Protect stored account data
- Requirement 3.4: Render PAN (Primary Account Number) unreadable anywhere it is stored
- Requirement 7: Restrict access to system components and cardholder data by business need to know

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.credit_accounts (
  account_id       INT,
  customer_name    STRING,
  credit_card_number STRING,
  ssn              STRING,
  email            STRING,
  phone            STRING,
  income           DOUBLE,
  credit_score     INT,
  account_balance  DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.credit_accounts VALUES
  (1, 'Alice Johnson',  '4532-1234-5678-9010', '123-45-6789', 'alice@example.com',  '555-0101', 85000.00,  720, 5420.50),
  (2, 'Bob Smith',      '5425-2334-3010-9288', '234-56-7890', 'bob@example.com',    '555-0202', 62000.00,  680, 12300.75),
  (3, 'Carol Evans',   '3714-496353-98431',   '345-67-8901', 'carol@example.com',  '555-0303', 145000.00, 790, 890.20),
  (4, 'David Garcia',  '4916-3289-1234-5678', '456-78-9012', 'david@example.com',  '555-0404', 38000.00,  610, 33100.00),
  (5, 'Eva Harris',    '5105-1051-0510-5100', '567-89-0123', 'eva@example.com',    '555-0505', 210000.00, 810, 2250.80),
  (6, 'Frank Iyer',    '4111-1111-1111-1111', '678-90-1234', 'frank@example.com',  '555-0606', 77000.00,  745, 18900.00)
""")


In [ ]:
spark.sql(f"""
-- Tag columns with PCI data classification
ALTER TABLE {catalog}.{schema}.credit_accounts
  ALTER COLUMN credit_card_number SET TAGS ('pci_data' = 'credit_card')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.credit_accounts
  ALTER COLUMN ssn SET TAGS ('pci_data' = 'ssn')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.credit_accounts
  ALTER COLUMN income SET TAGS ('pci_data' = 'income')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.credit_accounts
  ALTER COLUMN phone SET TAGS ('pci_data' = 'phone')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.credit_accounts
  ALTER COLUMN email SET TAGS ('pci_data' = 'email')
""")


### PCI-DSS Masking Function 1: Credit Card Tokenization

PCI-DSS Requirement 3.4 allows showing the first 6 and last 4 digits of a PAN. We implement the safer last-4-only format: `****-****-****-XXXX`.

In [ ]:
spark.sql(f"""
-- Credit card masking: show only last 4 digits in PCI-compliant format
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_credit_card(val STRING)
RETURNS STRING
RETURN CONCAT('****-****-****-', RIGHT(REPLACE(val, '-', ''), 4))
""")


### PCI-DSS Masking Function 2: SSN Format Preservation

Mask SSN while preserving the format pattern for usability in partial identity verification.

In [ ]:
spark.sql(f"""
-- SSN masking: XXX-XX-last4 format
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_ssn_finance(val STRING)
RETURNS STRING
RETURN CONCAT('XXX-XX-', RIGHT(val, 4))
""")


### PCI-DSS Masking Function 3: Income Bracket Generalization

Instead of redacting income entirely, we generalize it to brackets. This preserves analytical utility (e.g., segmentation) while preventing precise income disclosure.

In [ ]:
spark.sql(f"""
-- Income bracket masking: generalize to ranges for analytics use cases
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_income_bracket(val DOUBLE)
RETURNS STRING
RETURN CASE
  WHEN val < 25000   THEN '$0-25K'
  WHEN val < 50000   THEN '$25K-50K'
  WHEN val < 100000  THEN '$50K-100K'
  WHEN val < 250000  THEN '$100K-250K'
  ELSE '$250K+'
END
""")


### PCI-DSS Column Mask Policies

Apply three separate policies matching on specific `pci_data` tag values. Using `has_tag_value()` ensures each policy targets only its designated PCI data type.

In [ ]:
spark.sql(f"""
-- Policy 1: Mask credit card numbers
CREATE OR REPLACE POLICY pci_mask_credit_card
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_credit_card
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('pci_data', 'credit_card') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Policy 2: Mask SSN columns in finance context
CREATE OR REPLACE POLICY pci_mask_ssn
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_ssn_finance
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('pci_data', 'ssn') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Policy 3: Generalize income to brackets
-- Note: income is DOUBLE, mask returns STRING — cast in the UDF return type
CREATE OR REPLACE POLICY pci_mask_income
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_income_bracket
TO `account users`
EXCEPT abac_tut_admins, finance_ops
FOR TABLES
MATCH COLUMNS has_tag_value('pci_data', 'income') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Verify masking:
-- credit_card_number: ****-****-****-XXXX
-- ssn: XXX-XX-XXXX
-- income: bracket string
-- account_id, credit_score, account_balance: visible (not tagged)
SELECT account_id, customer_name, credit_card_number, ssn, income, credit_score, account_balance
FROM {catalog}.{schema}.credit_accounts
""")


---
## Healthcare — HIPAA PHI Column Masking

HIPAA's Privacy Rule defines 18 categories of Protected Health Information (PHI). Any combination of these that could identify an individual must be masked or de-identified before sharing with unauthorized parties.

**Key HIPAA requirements addressed:**
- 45 CFR § 164.514(b): Safe Harbor de-identification method
- 45 CFR § 164.502(b): Minimum Necessary standard
- PHI categories covered: names, dates (DOB), identification numbers, geographic data, contact information

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.patient_records (
  patient_id    INT,
  patient_name  STRING,
  dob           DATE,
  ssn           STRING,
  diagnosis     STRING,
  insurance_id  STRING,
  address       STRING,
  phone         STRING,
  email         STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.patient_records VALUES
  (1001, 'Alice Brown',    '1978-04-12', '123-45-6789', 'Hypertension',       'INS-001-AB', '123 Oak St, Boston MA',    '617-555-0101', 'alice.b@email.com'),
  (1002, 'Bob Davis',      '1965-09-30', '234-56-7890', 'Type 2 Diabetes',    'INS-002-BD', '456 Elm Ave, Chicago IL',  '312-555-0202', 'bob.d@email.com'),
  (1003, 'Carol Evans',    '1990-02-14', '345-67-8901', 'Asthma',             'INS-003-CE', '789 Pine Rd, Austin TX',   '512-555-0303', 'carol.e@email.com'),
  (1004, 'David Garcia',   '1955-11-22', '456-78-9012', 'Coronary Artery Disease', 'INS-004-DG', '321 Maple Dr, LA CA', '213-555-0404', 'david.g@email.com'),
  (1005, 'Eva Harris',     '1983-07-08', '567-89-0123', 'Hypothyroidism',     'INS-005-EH', '654 Cedar Ln, Seattle WA', '206-555-0505', 'eva.h@email.com'),
  (1006, 'Frank Iyer',     '1972-03-25', '678-90-1234', 'Migraine',           'INS-006-FI', '987 Birch Blvd, NYC NY',   '212-555-0606', 'frank.i@email.com')
""")


In [ ]:
spark.sql(f"""
-- Tag PHI columns with classification levels
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN patient_name SET TAGS ('phi_level' = 'name')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN dob SET TAGS ('phi_level' = 'dob')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN ssn SET TAGS ('phi_level' = 'id')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN insurance_id SET TAGS ('phi_level' = 'id')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN address SET TAGS ('phi_level' = 'contact')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN phone SET TAGS ('phi_level' = 'contact')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN email SET TAGS ('phi_level' = 'contact')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.patient_records
  ALTER COLUMN diagnosis SET TAGS ('phi_level' = 'diagnosis')
""")


### HIPAA PHI Masking Function 1: Name De-identification

HIPAA Safe Harbor requires removing names. We replace with first initial + `****` to indicate de-identification while preserving minimal context.

In [ ]:
spark.sql(f"""
-- PHI name masking: first initial + **** 
-- e.g., 'Alice Brown' -> 'A****'
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_phi_name(val STRING)
RETURNS STRING
RETURN CONCAT(LEFT(val, 1), '****')
""")


### HIPAA PHI Masking Function 2: DOB Age Range

HIPAA requires removing specific dates (including DOB). We replace DOB with an age range to preserve usefulness for cohort analysis while satisfying de-identification requirements. HIPAA requires ages >= 90 to be grouped into a single category.

In [ ]:
spark.sql(f"""
-- PHI DOB masking: convert to age range string
-- HIPAA requires grouping ages 90+ together
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_dob_age_range(dob DATE)
RETURNS STRING
RETURN (
  SELECT CASE
    WHEN age >= 90 THEN '90+ years'
    ELSE CONCAT(CAST((age DIV 5) * 5 AS STRING), '-', CAST((age DIV 5) * 5 + 4 AS STRING), ' years')
  END
  FROM (SELECT FLOOR(DATEDIFF(CURRENT_DATE(), dob) / 365.25) AS age)
)
""")


### HIPAA PHI Masking Function 3: Deterministic ID Hash

For ID columns (SSN, insurance ID), a SHA2 hash provides deterministic pseudonymization. The same input always produces the same hash, preserving referential integrity for joins across de-identified datasets.

In [ ]:
spark.sql(f"""
-- PHI ID masking: deterministic SHA2 hash preserves referential integrity
-- Same input always yields same hash — allows joins across de-identified tables
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_phi_id(val STRING)
RETURNS STRING
RETURN CONCAT('ID-', LEFT(SHA2(val, 256), 16))
""")


### HIPAA PHI Column Mask Policies

Four policies targeting different PHI categories. Clinical staff are exempt from the diagnosis masking policy so they can perform clinical work.

In [ ]:
spark.sql(f"""
-- Policy 1: Mask patient names
CREATE OR REPLACE POLICY hipaa_mask_name
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_phi_name
TO `account users`
EXCEPT abac_tut_admins, clinical_staff
FOR TABLES
MATCH COLUMNS has_tag_value('phi_level', 'name') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Policy 2: Mask DOB with age range
CREATE OR REPLACE POLICY hipaa_mask_dob
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_dob_age_range
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('phi_level', 'dob') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Policy 3: Hash ID columns (SSN, insurance_id)
CREATE OR REPLACE POLICY hipaa_mask_ids
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_phi_id
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('phi_level', 'id') AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Policy 4: Redact contact info and diagnosis for non-clinical users
CREATE OR REPLACE FUNCTION {catalog}.{schema}.redact_phi(val STRING)
RETURNS STRING
RETURN '***PHI-REDACTED***'
""")

spark.sql(f"""
CREATE OR REPLACE POLICY hipaa_mask_contact
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.redact_phi
TO `account users`
EXCEPT abac_tut_admins, clinical_staff, billing_staff
FOR TABLES
MATCH COLUMNS (has_tag_value('phi_level', 'contact')
  OR has_tag_value('phi_level', 'diagnosis')) AS m
ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Verify PHI masking:
-- patient_name: A****
-- dob: age range
-- ssn, insurance_id: ID-<hash>
-- address, phone, email, diagnosis: ***PHI-REDACTED***
-- patient_id: visible (internal ID only, not PHI by itself)
SELECT patient_id, patient_name, dob, ssn, diagnosis, insurance_id, address
FROM {catalog}.{schema}.patient_records
""")


**Expected Results by Role:**

| Role | patient_name | dob | ssn | diagnosis | contact fields |
|------|-------------|-----|-----|-----------|----------------|
| `abac_tut_admins` | Alice Brown | 1978-04-12 | 123-45-6789 | Hypertension | Full values |
| `clinical_staff` | Alice Brown | 45-49 years | ID-abc123... | Hypertension | ***PHI-REDACTED*** |
| `billing_staff` | A**** | 45-49 years | ID-abc123... | ***PHI-REDACTED*** | Full contact |
| Standard user | A**** | 45-49 years | ID-abc123... | ***PHI-REDACTED*** | ***PHI-REDACTED*** |